# 02 — Python ML evaluation evidence
This notebook validates certified Python handoffs and reads completed independent Python and Spark results. Training remains disabled to preserve completed runs. The original delay comparison uses the approved comparison-only UTC+05:00 normalization; source predictions remain unchanged.

Other completed task evidence and any unresolved SRS coverage gaps are read from the closure summary below. A Spark export cannot serve as independent Python evidence. Never point this notebook at raw production data.

See [ML methodology](../documentation/ML_EXECUTION_AND_COMPARISON.md) and [handoff contract](../ml_execution/README.md).

In [1]:
from pathlib import Path
import sys, json, hashlib, platform
REPO = Path('/home/manal/Desktop/UrbanTransit-IQ').resolve()
assert (REPO / 'feature_contracts.py').is_file(), 'Set REPO to the authoritative checkout'
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
def small_json(path, limit=2_000_000):
    path = Path(path).resolve()
    if 'raw_data' in path.parts: raise ValueError('Processed evidence only')
    with path.open('rb') as stream: payload = stream.read(limit + 1)
    if len(payload) > limit: raise ValueError('JSON exceeds notebook evidence size limit')
    return json.loads(payload)
print({'python': platform.python_version(), 'repo': str(REPO),
       'feature_contract_sha256': hashlib.sha256((REPO/'feature_contracts.py').read_bytes()).hexdigest()})


{'python': '3.10.12', 'repo': '/home/manal/Desktop/UrbanTransit-IQ', 'feature_contract_sha256': '2450437b27868be67361b78944ce7bb1c8bd575dab77b3a0e5e108a2447b13d1'}


## 1. Preserved delay evidence and disabled training
Repeat this notebook for each certified task/grain. Tasks: delay severity, passenger demand,
occupancy forecast, crowding risk and route clustering. Descriptive service analytics is
separate from model evaluation. Limits below are notebook safeguards, not evidence of scale.


In [2]:
FEATURE_MANIFEST = REPO / 'reports/ml_handoffs/python-delay-severity-prod-v11-20260928T1935Z/manifest.json'
COMPLETED_PYTHON_RESULT = REPO / 'reports/ml_runs/python-delay-severity-prod-v11-20260928T2030Z/result.json'
COMPLETED_SPARK_RESULT = REPO / 'reports/ml_runs/spark-delay-severity-prod-v11-20260928T1630Z/result.json'
COMPLETED_COMPARISON = REPO / 'reports/ml_comparisons/stage=spark_python_comparison/dataset=production-v1.1/run=python-delay-v11-normalized-20260928T2100Z/comparison.json'
NEW_PYTHON_RUN = None  # PENDING: fresh separate run directory, never an active output
VERIFY_FEATURE_PACKAGE = True
RUN_PYTHON_TRAINING = False  # Leave False during production; enable only in a later approved ML run
READ_COMPLETED_RESULTS = True
MAX_CASES = 100000
MAX_PACKAGE_BYTES = 128 * 1024**2
manifest, partitions, input_hashes = None, None, None
python_result = None


## 2. Certified handoff, finite predictors and leakage validation
The existing loader verifies manifest/evidence hashes, row uniqueness, departure isolation,
chronological boundaries, label and predictor availability, TRAIN class coverage and capacity
eligibility. It rejects missing/uncertified packages. No imputation, label derivation, random
partitioning or raw-data loading is performed here. Review predictor meanings/aliases manually too.


In [3]:
if VERIFY_FEATURE_PACKAGE:
    from ml_execution.contracts import validate_manifest, load_features
    if FEATURE_MANIFEST is None: raise ValueError('PENDING certified Python task manifest')
    path = Path(FEATURE_MANIFEST).resolve()
    candidate = validate_manifest(small_json(path))
    if candidate['producer'] != 'python': raise ValueError('Independent Python features required')
    root = path.parent
    package_files = [(root / candidate['certification']['evidence_path']).resolve()]
    package_files += [(root / spec['path']).resolve() for spec in candidate['partitions'].values()]
    if any(not p.is_relative_to(root) or 'raw_data' in p.parts for p in package_files):
        raise ValueError('All inputs must stay in the processed feature package')
    if sum(p.stat().st_size for p in package_files) > MAX_PACKAGE_BYTES:
        raise ValueError('Request an approved smaller certified cohort; do not truncate partitions')
    manifest, partitions, input_hashes = load_features(path, max_rows=MAX_CASES)
    display({'task':manifest['task_name'], 'dataset':manifest['dataset_version'],
             'features':manifest['features'], 'grain':manifest['grain'],
             'counts':{k:len(v) for k,v in partitions.items()}, 'hashes':input_hashes})
else:
    print('PENDING: certified bounded independent Python feature package')


{'task': 'delay_severity',
 'dataset': 'production-v1.1',
 'features': ['scheduled_hour_utc',
  'service_weekday',
  'is_weekend',
  'service_month',
  'direction_id',
  'distance_km'],
 'grain': 'one valid observed trip stop event derived from certified raw Python inputs',
 'counts': {'test': 16588, 'train': 66883, 'validation': 16529},
 'hashes': {'manifest_sha256': '158a6ca9d96e9423fb31f75a55573704fdba8296abf4bc6e617c28daa16d5315',
  'partition_sha256': {'test': 'aba5cc17c9d845df656b1565f99c98e6719571d953f75d04880fc91911d13229',
   'train': '909340b4372032e565d3c8158612ddeae8a4b87f617d04c7edee5de219a28b8c',
   'validation': '353d71e9cf2da8b87efaa3800e4834015a1f356d7d42d952f629594e07d28fb1'}}}

## 3. Frozen evaluation protocol and candidate selection
Fit preprocessing/models on TRAIN (2025) only. Select on VALIDATION (2026-Q1);
evaluate the frozen winner on TEST (2026-Q2). Never use TEST for feature choice or tuning.
The runner compares three classifiers/regressors or K-means k=2,3,4. Select classifiers
by macro F1, regressors by MAE, clustering by silhouette, with alphabetical tie-break.
All three candidate scores must be valid. There is no TRAIN+VALIDATION refit.
Forecasts use certified rolling-origin lag observations with frozen model parameters.
Classification reports accuracy, weighted and macro scores, class support/confusion matrix;
forecasting reports MAE/RMSE/MAPE/R² plus the certified historical baseline on identical cases.
MAPE excludes zero actuals and reports counts; undefined scores remain null.
Measured results and acceptance outcomes are read from persisted result artifacts below.


In [4]:


if RUN_PYTHON_TRAINING:
    if not VERIFY_FEATURE_PACKAGE or manifest is None:
        raise ValueError('Validate the certified package first')
    if NEW_PYTHON_RUN is None: raise ValueError('Choose a new separate run directory')
    output = Path(NEW_PYTHON_RUN).resolve()
    if output.exists() or any(p in output.parts for p in ('raw_data', 'production-v1', 'production-v1.1')):
        raise ValueError('Refusing existing or protected output')
    from ml_execution.runner import execute
    training_result = execute(FEATURE_MANIFEST, output, engine='python', max_rows=MAX_CASES)
    print({'status':training_result['status'], 'reason':training_result.get('reason'), 'output':str(output)})
else:
    print('NOT RUN: Python training disabled; no Spark execution exists in this notebook')


NOT RUN: Python training disabled; no Spark execution exists in this notebook


## 4. Genuine results, candidates, metrics and saved artifacts
Read only a completed small result artifact. Fixture, missing and incomplete results do not
satisfy production acceptance. Inspect failed candidates, sample counts, versions, hashes,
parameters, timings, baseline comparisons and acceptance separately from execution success.
Saved model reload checks are recorded in the recovery audit; this notebook reads that evidence without unpickling models.


In [5]:
from ml_execution.results import read_result
def completed_result(path, engine):
    if path is None: return None
    small_json(path)  # size/raw-path guard before the project's contract reader
    result = read_result(path)
    if result.get('status') != 'SUCCEEDED':
        print(result); return None
    if result.get('engine') != engine or result.get('task_contract', {}).get('producer') != engine:
        raise ValueError('Wrong independent producer')
    return result
if READ_COMPLETED_RESULTS:
    python_result = completed_result(COMPLETED_PYTHON_RESULT, 'python')
if python_result is None:
    print('PENDING: measured Python candidates, model winner and held-out metrics')
else:
    for key in ('task_name','sample_counts','periods','model_comparison','selected_model',
                'selection_rationale','metrics','baseline_metrics','baseline_vs_selected',
                'acceptance','runtime_sec','artifact_paths','input_hashes','reproducibility'):
        display({key:python_result.get(key)})


{'task_name': 'delay_severity'}

{'sample_counts': {'test': 16588, 'train': 66883, 'validation': 16529}}

{'periods': {'test': {'end': '2026-06-30', 'start': '2026-04-01'},
  'train': {'end': '2025-12-31', 'start': '2025-01-01'},
  'validation': {'end': '2026-03-31', 'start': '2026-01-01'}}}

{'model_comparison': [{'effective_parameters': {'estimator__C': 1.0,
    'estimator__class_weight': None,
    'estimator__dual': False,
    'estimator__fit_intercept': True,
    'estimator__intercept_scaling': 1,
    'estimator__l1_ratio': None,
    'estimator__max_iter': 300,
    'estimator__multi_class': 'deprecated',
    'estimator__n_jobs': None,
    'estimator__penalty': 'l2',
    'estimator__random_state': 20260926,
    'estimator__solver': 'lbfgs',
    'estimator__tol': 0.0001,
    'estimator__verbose': 0,
    'estimator__warm_start': False,
    'memory': None,
    'scale__copy': True,
    'scale__with_mean': True,
    'scale__with_std': True,
    'transform_input': None,
    'verbose': False},
   'model_name': 'logistic_regression',
   'model_type': 'classification',
   'parameters': {'max_iter': 300},
   'random_seed': 20260926,
   'runtime_sec': 3.382421614998748,
   'status': 'SUCCEEDED',
   'validation_metrics': {'accuracy': 0.7313207090568092,
    'averaging': 'support_wei

{'selected_model': 'decision_tree'}

...

{'acceptance': {'accuracy': 0.7336026042922594,
  'macro_f1': 0.5456510424522178,
  'passed': False,
  'requirement': 'test accuracy >= 0.85 OR macro F1 >= 0.80',
  'status': 'MEASURED'}}

{'runtime_sec': 13.61648636999962}

{'artifact_paths': {'model': '/home/manal/Desktop/UrbanTransit-IQ/reports/ml_runs/python-delay-severity-prod-v11-20260928T2030Z/selected_model.pkl',
  'predictions': '/home/manal/Desktop/UrbanTransit-IQ/reports/ml_runs/python-delay-severity-prod-v11-20260928T2030Z/predictions.jsonl',
  'result': '/home/manal/Desktop/UrbanTransit-IQ/reports/ml_runs/python-delay-severity-prod-v11-20260928T2030Z/result.json'}}

{'input_hashes': {'manifest_sha256': '158a6ca9d96e9423fb31f75a55573704fdba8296abf4bc6e617c28daa16d5315',
  'partition_sha256': {'test': 'aba5cc17c9d845df656b1565f99c98e6719571d953f75d04880fc91911d13229',
   'train': '909340b4372032e565d3c8158612ddeae8a4b87f617d04c7edee5de219a28b8c',
   'validation': '353d71e9cf2da8b87efaa3800e4834015a1f356d7d42d952f629594e07d28fb1'}}}

{'reproducibility': {'command': ['/home/manal/Desktop/UrbanTransit-IQ/ml_execution/__main__.py',
   '--manifest',
   'reports/ml_handoffs/python-delay-severity-prod-v11-20260928T1935Z/manifest.json',
   '--output',
   'reports/ml_runs/python-delay-severity-prod-v11-20260928T2030Z',
   '--engine',
   'python',
   '--max-rows',
   '100000'],
  'created_at_utc': '2026-09-28T14:33:17.507925+00:00',
  'libraries': {'numpy': '1.26.4', 'pyspark': None, 'scikit-learn': '1.7.2'},
  'platform': 'Linux-6.8.0-138-generic-x86_64-with-glibc2.35',
  'python': '3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]',
  'source_sha256': {'ml_execution/__init__.py': 'a4c084b9908f2c28c9836e5a2e74adb5ebb07d3727f0faeb280f9f7f7642d711',
   'ml_execution/__main__.py': '779f9ba5b96ee8f5a3abb8a050a4d0b452bcf0e74e258f86bd444b5f8ec98817',
   'ml_execution/analytics.py': '91d348317fad956e6bd61552650de521946ef621058456f03cac921b4b8f423a',
   'ml_execution/clustering_handoff.py': '316791b157c67f41594c82e10bf6e8d92cbed7

## 5. Independent dual-pipeline evidence readiness
Matching task/dataset alone does not prove equivalence or independence. The comparison stage
must align at least 100 unique unseen TEST cases with matching truth, target windows and
operational identity, retain separate producer/feature hashes and report accuracy/error,
runtime, differences and limitations. The recorded production-v1.1 delay report uses an
explicit comparison-only UTC+05:00 correction; source result and prediction artifacts remain unchanged.


In [6]:
spark_result = completed_result(COMPLETED_SPARK_RESULT, 'spark') if READ_COMPLETED_RESULTS else None
if python_result is None or spark_result is None:
    print('PENDING: both independent successful result and prediction artifacts')
else:
    for key in ('task_name', 'dataset_version', 'periods'):
        if python_result[key] != spark_result[key]: raise ValueError(f'Comparison mismatch: {key}')
    display({engine:{key:r.get(key) for key in ('feature_version','input_hashes','sample_counts','runtime_sec')}
             for engine,r in [('python',python_result),('spark',spark_result)]})
    comparison_result = small_json(COMPLETED_COMPARISON)
    if comparison_result.get('status') != 'SUCCEEDED': raise ValueError('The recorded dual-pipeline comparison did not succeed')
    display({'comparison_status':comparison_result['status'], 'comparison_cases':comparison_result['comparison_case_count'],
             'metrics':comparison_result['metrics'], 'alignment':comparison_result['alignment'],
             'input_sha256':comparison_result['input_sha256']})


{'python': {'feature_version': 'python-raw-delay-1607c910dd6dbe59759dd7a8ceb955cc2b44f31739a4bbd655ef654b95036cdd',
  'input_hashes': {'manifest_sha256': '158a6ca9d96e9423fb31f75a55573704fdba8296abf4bc6e617c28daa16d5315',
   'partition_sha256': {'test': 'aba5cc17c9d845df656b1565f99c98e6719571d953f75d04880fc91911d13229',
    'train': '909340b4372032e565d3c8158612ddeae8a4b87f617d04c7edee5de219a28b8c',
    'validation': '353d71e9cf2da8b87efaa3800e4834015a1f356d7d42d952f629594e07d28fb1'}},
  'sample_counts': {'test': 16588, 'train': 66883, 'validation': 16529},
  'runtime_sec': 13.61648636999962},
 'spark': {'feature_version': 'spark-spark-prod-v11-20260928T0944Z-1a7c3f-delay_severity',
  'input_hashes': {'manifest_sha256': '21f10fd591b1cb98ac6b3f7004c2bf3260adb131ec88db32443a0ac38adb9446',
   'partition_sha256': {'test': '2bf41329fd13c3c37e147678bd54e8307d5911593e2799f46bd744fa27408fb0',
    'train': 'ccaca7f358650ad886aa126670dac3a2073ce1feb44951ff18d53fecdaa78558',
    'validation': 'ff

{'comparison_status': 'SUCCEEDED',
 'comparison_cases': 2429,
 'metrics': {'agreement_rate': 0.9971181556195965,
  'matching_predictions': 2422,
  'mismatching_predictions': 7,
  'numeric_delta_max_abs': 2.0,
  'numeric_delta_mean': 0.005763688760806916,
  'python_only_test_cases': 14159,
  'python_test_cases': 16588,
  'shared_test_cases': 2429,
  'spark_only_test_cases': 14159,
  'spark_test_cases': 16588},
 'alignment': {'case_identity': 'stop_event_id extracted from Spark length-prefixed trip_id + stop_event_id',
  'policy': 'production-v1.1-delay-source-event-utc5',
  'python_only_test_cases': 14159,
  'python_test_cases': 16588,
  'scope': 'comparison only; source result and prediction artifacts are unchanged',
  'shared_test_cases': 2429,
  'spark_only_test_cases': 14159,
  'spark_target_timestamp_correction_hours': -5,
  'spark_test_cases': 16588,
  'timestamp_basis': 'Asia/Karachi fixed UTC+05:00; normalize Spark target timestamps to UTC'},
 'input_sha256': {'python_prediction

In [7]:
if python_result is not None and spark_result is not None:
    import datetime as dt
    handoff_test = {}
    with (FEATURE_MANIFEST.parent / 'test.jsonl').open(encoding='utf-8') as stream:
        for line in stream:
            row = json.loads(line)
            handoff_test[row['source_stop_event_id']] = row
    spark_test = {}
    with Path(spark_result['artifact_paths']['predictions']).open(encoding='utf-8') as stream:
        for line in stream:
            row = json.loads(line)
            if row.get('split') != 'test': continue
            case_id = row['case_id']; colon = case_id.index(':')
            trip_length = int(case_id[:colon])
            event_id = case_id[colon + 1 + trip_length:]
            spark_test[event_id] = row
    common = handoff_test.keys() & spark_test.keys()
    target_matches = sum(handoff_test[key]['target_start'] == spark_test[key]['target_start'] for key in common)
    normalized_target_matches = sum(dt.datetime.fromisoformat(handoff_test[key]['target_start'].replace('Z', '+00:00')) == (dt.datetime.fromisoformat(spark_test[key]['target_start'].replace('Z', '+00:00')) - dt.timedelta(hours=5)) for key in common)
    truth_matches = sum(handoff_test[key]['target'] == spark_test[key]['actual'] for key in common)
    offsets = sorted({round((dt.datetime.fromisoformat(handoff_test[key]['target_start'].replace('Z', '+00:00')) - dt.datetime.fromisoformat(spark_test[key]['target_start'].replace('Z', '+00:00'))).total_seconds() / 3600, 3) for key in common})
    display({'python_test_rows':len(handoff_test), 'spark_test_rows':len(spark_test),
             'shared_source_event_ids':len(common), 'matching_truth':truth_matches,
             'matching_raw_target_timestamps':target_matches, 'matching_after_approved_utc5_normalization':normalized_target_matches,
             'python_minus_spark_target_hours':offsets,
             'comparison_ready':len(common) >= 100 and truth_matches == len(common) and normalized_target_matches == len(common)})
else:
    print('Comparison audit not run: both certified measured results are required.')


{'python_test_rows': 16588,
 'spark_test_rows': 16588,
 'shared_source_event_ids': 2429,
 'matching_truth': 2429,
 'matching_raw_target_timestamps': 0,
 'matching_after_approved_utc5_normalization': 2429,
 'python_minus_spark_target_hours': [-5.0],
 'comparison_ready': True}

## 6. Mandatory model evidence and remaining coverage
The following cells read the final task inventory, measured metrics, acceptance results, model paths, reload checks, comparison evidence and known limitations. Execution success and acceptance are separate. Failed thresholds are retained.

In [8]:
CLOSURE_PATH = REPO / 'reports/recovery/ml-closure-final-20260928-v2/summary.json'
closure = small_json(CLOSURE_PATH)
display({k: closure[k] for k in ('status', 'complete', 'partial', 'blocked', 'not_required', 'limitations', 'additional_tasks')})


{'status': 'FIVE_TASK_EXECUTION_COMPLETE_WITH_SRS_GAPS',
 'complete': ['delay_severity: spark',
  'delay_severity: python',
  'passenger_demand: spark',
  'passenger_demand: python',
  'occupancy_forecast: spark',
  'occupancy_forecast: python',
  'crowding_risk: spark',
  'crowding_risk: python',
  'route_clustering: spark',
  'route_clustering: python'],
 'partial': ['Python delay_service_analytics has code and tests but no certified production handoff/result; full SRS analytics coverage is not established.',
  'Anomaly detection and descriptive service analytics have code/tests but are outside the five fitted model runs; full production SRS analytical coverage is not established.',
  'Demand model evidence is at scheduled-trip grain. Separate stop, route, day/time-period forecast coverage and configurable future horizons are not established by these runs.',
  'Human interpretation/sign-off, comparison PDF and application integration remain pending.'],
 'blocked': ['HDFS namespace/fe

In [9]:
for task in closure['tasks']:
    print(task['task_name'])
    for engine in ('spark', 'python'):
        item = task[engine]
        result = completed_result(REPO / item['result_path'], engine)
        assert result is not None
        display({engine: {key: result.get(key) for key in ('selected_model', 'model_comparison', 'sample_counts', 'metrics', 'acceptance', 'baseline_vs_selected', 'artifact_paths')}})
    manifest_path = REPO / task['python']['manifest_path']
    checked_manifest, checked_rows, checked_hashes = load_features(manifest_path, max_rows=200000)
    assert checked_manifest['producer'] == 'python'
    assert checked_hashes == completed_result(REPO / task['python']['result_path'], 'python')['input_hashes']
    display({'python_handoff': str(manifest_path), 'rows': {s: len(rows) for s, rows in checked_rows.items()}, 'hashes': checked_hashes})
    del checked_rows


delay_severity


{'spark': {'selected_model': 'random_forest',
  'model_comparison': [{'effective_parameters': {'stage0.inputCol': 'raw_features',
     'stage0.outputCol': 'features',
     'stage0.withMean': True,
     'stage0.withStd': True,
     'stage1.aggregationDepth': 2,
     'stage1.elasticNetParam': 0.0,
     'stage1.family': 'auto',
     'stage1.featuresCol': 'features',
     'stage1.fitIntercept': True,
     'stage1.labelCol': 'label',
     'stage1.maxBlockSizeInMB': 0.0,
     'stage1.maxIter': 300,
     'stage1.predictionCol': 'prediction',
     'stage1.probabilityCol': 'probability',
     'stage1.rawPredictionCol': 'rawPrediction',
     'stage1.regParam': 0.0,
     'stage1.standardization': True,
     'stage1.threshold': 0.5,
     'stage1.tol': 1e-06},
    'model_name': 'logistic_regression',
    'model_type': 'classification',
    'parameters': {'max_iter': 300},
    'random_seed': 20260926,
    'runtime_sec': 158.39530291000028,
    'status': 'SUCCEEDED',
    'validation_metrics': {'accur

{'python': {'selected_model': 'decision_tree',
  'model_comparison': [{'effective_parameters': {'estimator__C': 1.0,
     'estimator__class_weight': None,
     'estimator__dual': False,
     'estimator__fit_intercept': True,
     'estimator__intercept_scaling': 1,
     'estimator__l1_ratio': None,
     'estimator__max_iter': 300,
     'estimator__multi_class': 'deprecated',
     'estimator__n_jobs': None,
     'estimator__penalty': 'l2',
     'estimator__random_state': 20260926,
     'estimator__solver': 'lbfgs',
     'estimator__tol': 0.0001,
     'estimator__verbose': 0,
     'estimator__warm_start': False,
     'memory': None,
     'scale__copy': True,
     'scale__with_mean': True,
     'scale__with_std': True,
     'transform_input': None,
     'verbose': False},
    'model_name': 'logistic_regression',
    'model_type': 'classification',
    'parameters': {'max_iter': 300},
    'random_seed': 20260926,
    'runtime_sec': 3.382421614998748,
    'status': 'SUCCEEDED',
    'validati

{'python_handoff': '/home/manal/Desktop/UrbanTransit-IQ/reports/ml_handoffs/python-delay-severity-prod-v11-20260928T1935Z/manifest.json',
 'rows': {'test': 16588, 'train': 66883, 'validation': 16529},
 'hashes': {'manifest_sha256': '158a6ca9d96e9423fb31f75a55573704fdba8296abf4bc6e617c28daa16d5315',
  'partition_sha256': {'test': 'aba5cc17c9d845df656b1565f99c98e6719571d953f75d04880fc91911d13229',
   'train': '909340b4372032e565d3c8158612ddeae8a4b87f617d04c7edee5de219a28b8c',
   'validation': '353d71e9cf2da8b87efaa3800e4834015a1f356d7d42d952f629594e07d28fb1'}}}

passenger_demand


...

{'python_handoff': '/home/manal/Desktop/UrbanTransit-IQ/reports/ml_handoffs/python-occupancy-recovery-20260928-a/crowding_risk/manifest.json',
 'rows': {'test': 11587, 'train': 70016, 'validation': 18397},
 'hashes': {'manifest_sha256': 'b9506ada0b4a5a9ea5aa1224a1ef6eee91d7955a925a78eaa8bdbed2a0db6ff4',
  'partition_sha256': {'test': 'bea1ee26c3cc4b6723451c0dbdbca995b1c68d939de43288f6594ff95ec3d0e1',
   'train': '00c76773b1bb8b28d6e94b4081c39be0a8bc52b22c55f77b11a321310e61dcae',
   'validation': 'd55b86c5aaf3705ddd607730687dcfef263fd21e76ca02ad4609a0b40b1ce1b4'}}}

route_clustering


{'spark': {'selected_model': 'kmeans_k4',
  'model_comparison': [{'effective_parameters': {'stage0.inputCol': 'raw_features',
     'stage0.outputCol': 'features',
     'stage0.withMean': True,
     'stage0.withStd': True,
     'stage1.distanceMeasure': 'euclidean',
     'stage1.featuresCol': 'features',
     'stage1.initMode': 'k-means||',
     'stage1.initSteps': 2,
     'stage1.k': 2,
     'stage1.maxBlockSizeInMB': 0.0,
     'stage1.maxIter': 100,
     'stage1.predictionCol': 'prediction',
     'stage1.seed': 20260926,
     'stage1.solver': 'auto',
     'stage1.tol': 0.0001},
    'model_name': 'kmeans_k2',
    'model_type': 'clustering',
    'parameters': {'k': 2, 'max_iter': 100},
    'random_seed': 20260926,
    'runtime_sec': 17.243128672998864,
    'status': 'SUCCEEDED',
    'validation_metrics': {'distance': 'euclidean on independently train-standardized features',
     'sample_count': 110,
     'sample_seed': 20260926,
     'silhouette': 0.3847082435125439}},
   {'effective_pa

{'python': {'selected_model': 'kmeans_k4',
  'model_comparison': [{'effective_parameters': {'estimator__algorithm': 'lloyd',
     'estimator__copy_x': True,
     'estimator__init': 'k-means++',
     'estimator__max_iter': 100,
     'estimator__n_clusters': 2,
     'estimator__n_init': 10,
     'estimator__random_state': 20260926,
     'estimator__tol': 0.0001,
     'estimator__verbose': 0,
     'memory': None,
     'scale__copy': True,
     'scale__with_mean': True,
     'scale__with_std': True,
     'transform_input': None,
     'verbose': False},
    'model_name': 'kmeans_k2',
    'model_type': 'clustering',
    'parameters': {'k': 2, 'max_iter': 100},
    'random_seed': 20260926,
    'runtime_sec': 2.459158617000412,
    'status': 'SUCCEEDED',
    'validation_metrics': {'distance': 'euclidean on independently train-standardized features',
     'sample_count': 110,
     'sample_seed': 20260926,
     'silhouette': 0.3868601302038277}},
   {'effective_parameters': {'estimator__algorith

{'python_handoff': '/home/manal/Desktop/UrbanTransit-IQ/reports/ml_handoffs/python-route-clustering-prod-v11-20260928T2210Z/manifest.json',
 'rows': {'train': 110, 'validation': 110, 'test': 120},
 'hashes': {'manifest_sha256': 'c3e6f0c59e25a0b0fb4158c74a8ba0b8b3198965bff75151476c225e917a0e7d',
  'partition_sha256': {'train': '259604261ca41e16531042bbee9bbd17e1f3afc9a7721b0f3dea20421d79cbc4',
   'validation': 'ca2b48b0e0071b52d49aa7eebc20a5e384239a07eb163b73e092267eedbdd364',
   'test': '5729539aa72f571690935c312ea9c6cf0081b894bcdb3d3ede9a9d2ebf51a135'}}}

In [10]:
audit = small_json(REPO / closure['artifact_audit_path'])
display([{'result': row['result'], 'input_hashes_verified': row['input_hashes_verified'], 'reload_prediction_parity_cases_per_split': row.get('reload_prediction_parity_cases_per_split')} for row in audit['runs']])
case_evidence = small_json(REPO / closure['comparison_case_summary'])
assert case_evidence['case_count'] >= 100
display({k: case_evidence[k] for k in ('case_count', 'agreement_count', 'disagreements', 'limitations')})


[{'result': 'reports/ml_runs/python-crowding-recovery-20260928-a/result.json',
  'input_hashes_verified': True,
  'reload_prediction_parity_cases_per_split': 64},
 {'result': 'reports/ml_runs/python-delay-severity-prod-v11-20260928T2030Z/result.json',
  'input_hashes_verified': True,
  'reload_prediction_parity_cases_per_split': 64},
 {'result': 'reports/ml_runs/python-occupancy-recovery-20260928-a/result.json',
  'input_hashes_verified': True,
  'reload_prediction_parity_cases_per_split': 64},
 {'result': 'reports/ml_runs/python-passenger-demand-prod-v11-20260928T2110Z/result.json',
  'input_hashes_verified': True,
  'reload_prediction_parity_cases_per_split': 64},
 {'result': 'reports/ml_runs/python-route-clustering-prod-v11-20260928T2210Z/result.json',
  'input_hashes_verified': True,
  'reload_prediction_parity_cases_per_split': 64},
 {'result': 'reports/ml_runs/spark-crowding-risk-prod-v11-20260928T1810Z/result.json',
  'input_hashes_verified': True,
  'reload_prediction_parity_ca

{'case_count': 2429,
 'agreement_count': 2422,
 'disagreements': [{'case_id': 'TSE0c01b3cbc159f1cbc2b2e615f3be8725293e0f2d23b3d277c3e1f1ca47e3a469',
   'trip_id': 'TRe27b1dffe401022504e4b9e62dabed801ac5d2bb3be19c9b330f4986ff22513c',
   'operational_departure_id': 'OPD665512dbb59bb02a421d52bbc62540472a58332a2ff9b3bbad216b2df194389d',
   'service_date': '2026-04-30',
   'actual': 0,
   'python_prediction': 0,
   'spark_prediction': 2.0,
   'match': False,
   'difference_spark_minus_python': 2.0,
   'python_target_start': '2026-04-30T05:05:30Z',
   'spark_original_target_start': '2026-04-30T10:05:30Z',
   'comparison_normalization': {'case_identity': 'stop_event_id extracted from Spark length-prefixed trip_id + stop_event_id',
    'policy': 'production-v1.1-delay-source-event-utc5',
    'python_only_test_cases': 14159,
    'python_test_cases': 16588,
    'scope': 'comparison only; source result and prediction artifacts are unchanged',
    'shared_test_cases': 2429,
    'spark_only_test_ca